============================================
 
Kitsap Transit — StreetLight Data 
============================================
 
MODE SHARE ANALYSIS
  1. Choropleth — bike/ped mode share by tract, most recent year
  2. Choropleth — vehicle dominance by tract
     
=============================================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import warnings
warnings.filterwarnings('ignore')

=============================================================================
CONFIGURATION
=============================================================================

In [ ]:
INPUT_FILE  = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014403_ModeShare_Winslow\2014403_ModeShare_Winslow_active_transportation_bike_ped_veh.csv'
SHP_FILE    = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014403_ModeShare_Winslow\Shapefile\2014403_ModeShare_Winslow_tracts\2014403_ModeShare_Winslow_tracts.shp'
OUTPUT_DIR  = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\mode_share_maps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Convert full FIPS to short readable tract label e.g. 53035080101 -> 801.01
def fips_to_label(fips):
    s = str(fips).strip()
    if len(s) == 11:
        tract_raw = s[5:]           # last 6 digits
        main      = str(int(tract_raw[:4]))   # drop leading zeros from main
        sub       = tract_raw[4:]
        if sub == '00':
            return f'Tract {main}'
        else:
            return f'Tract {main}.{sub}'
    return str(fips)

=============================================================================
LOAD DATA
=============================================================================

In [ ]:
print("Loading data...")
df  = pd.read_csv(INPUT_FILE)
gdf = gpd.read_file(SHP_FILE)

In [ ]:
# Standardize tract ID columns
df['Census Tract ID']  = df['Census Tract ID'].astype(str).str.strip()
gdf['tract_id']        = gdf['name'].astype(str).str.strip()
gdf['tract_label']     = gdf['tract_id'].apply(fips_to_label)

In [ ]:
print(f"  CSV: {len(df):,} records  |  Shapefile: {len(gdf)} tracts")
print(f"  Years in data: {sorted(df['Year'].unique().tolist())}")
print(f"  Modes: {df['Mode of Travel'].unique().tolist()}\n")

In [ ]:
# Clean mode names
MODE_MAP = {
    'All Vehicles - StL All Vehicles Volume': 'vehicle',
    'Bicycle - StL Bicycle Volume':           'bicycle',
    'Pedestrian - StL Pedestrian Volume':     'pedestrian'
}
df['mode'] = df['Mode of Travel'].map(MODE_MAP)

=============================================================================
PIVOT TO WIDE FORMAT
=============================================================================

In [ ]:
wide = df.pivot_table(
    index=['Census Tract ID', 'Year'],
    columns='mode',
    values=['Mode Share', 'StreetLight Volume', 'Population Normalized Volume'],
    aggfunc='first'
).reset_index()

In [ ]:
# Flatten column names
wide.columns = ['_'.join(filter(None, col)).strip('_') if isinstance(col, tuple)
                else col for col in wide.columns]
wide.columns = wide.columns.str.replace(' ', '_')

In [ ]:
# Compute combined bike+ped share
wide['bikePed_share'] = (
    wide.get('Mode_Share_bicycle', 0).fillna(0) +
    wide.get('Mode_Share_pedestrian', 0).fillna(0)
)
wide['vehicle_share'] = wide.get('Mode_Share_vehicle', pd.Series(dtype=float)).fillna(0)

In [ ]:
wide['Census_Tract_ID'] = wide['Census_Tract_ID'].astype(str).str.strip()

=============================================================================
MERGE WITH GEOMETRY
=============================================================================

In [ ]:
def merge_year(year):
    subset = wide[wide['Year'] == year].copy()
    merged = gdf.merge(subset, left_on='tract_id', right_on='Census_Tract_ID', how='left')
    return merged

=============================================================================
SHARED MAP STYLING
=============================================================================

In [ ]:
def base_map(ax, title, note=None):
    ax.set_axis_off()
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    if note:
        ax.text(0.01, 0.01, note, transform=ax.transAxes,
                fontsize=7, color='#666666', va='bottom')

In [ ]:
def add_colorbar(fig, ax, cmap, vmin, vmax, label):
    sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(label, fontsize=9)
    return cbar

In [ ]:
def add_tract_labels(ax, gdf_plot, col='tract_label', fontsize=5.5):
    for _, row in gdf_plot.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue
        pt = row.geometry.centroid
        ax.annotate(row[col], xy=(pt.x, pt.y),
                    ha='center', va='center',
                    fontsize=fontsize, color='#333333',
                    bbox=dict(boxstyle='round,pad=0.1',
                              facecolor='white', alpha=0.5, linewidth=0))

=============================================================================
ANALYSIS 1: BIKE + PED MODE SHARE CHOROPLETH — MOST RECENT YEAR
=============================================================================

In [ ]:
print("Analysis 1: Bike + Ped mode share choropleth...")

In [ ]:
latest_year = int(wide['Year'].max())
gdf_latest  = merge_year(latest_year)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
gdf_latest.plot(
    column='bikePed_share',
    cmap='YlGn',
    linewidth=0.5,
    edgecolor='white',
    legend=False,
    missing_kwds={'color': '#eeeeee', 'label': 'No data'},
    ax=ax,
    vmin=0,
    vmax=gdf_latest['bikePed_share'].quantile(0.95)
)
add_colorbar(fig, ax, 'YlGn',
             vmin=0,
             vmax=gdf_latest['bikePed_share'].quantile(0.95),
             label='Bike + Pedestrian Mode Share (%)')
add_tract_labels(ax, gdf_latest)
base_map(ax,
         f'Bike + Pedestrian Mode Share by Census Tract — {latest_year}\nBainbridge Island & Kitsap County',
         note='Source: StreetLight Active Transportation Analysis  |  Darker = higher non-vehicle mode share')

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'BikePed_ModeShare_{latest_year}.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved BikePed_ModeShare_{latest_year}.pdf")

=============================================================================
ANALYSIS 2: VEHICLE DOMINANCE CHOROPLETH — MOST RECENT YEAR
=============================================================================

In [ ]:
print("Analysis 2: Vehicle dominance choropleth...")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
gdf_latest.plot(
    column='vehicle_share',
    cmap='RdYlGn_r',   # red = high vehicle dominance
    linewidth=0.5,
    edgecolor='white',
    legend=False,
    missing_kwds={'color': '#eeeeee'},
    ax=ax,
    vmin=gdf_latest['vehicle_share'].quantile(0.05),
    vmax=100
)
add_colorbar(fig, ax, 'RdYlGn_r',
             vmin=gdf_latest['vehicle_share'].quantile(0.05),
             vmax=100,
             label='Vehicle Mode Share (%)')
add_tract_labels(ax, gdf_latest)
base_map(ax,
         f'Vehicle Mode Dominance by Census Tract — {latest_year}\nBainbridge Island & Kitsap County',
         note='Source: StreetLight Active Transportation Analysis  |  Red = highest vehicle dominance  Green = lower vehicle share')

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'Vehicle_Dominance_{latest_year}.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved Vehicle_Dominance_{latest_year}.pdf")

=============================================================================
ANALYSIS 3: SIDE-BY-SIDE — BIKE/PED vs VEHICLE DOMINANCE
=============================================================================

In [ ]:
print("Analysis 3: Side-by-side comparison...")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 12))

In [ ]:
# Left — bike/ped
gdf_latest.plot(
    column='bikePed_share', cmap='YlGn',
    linewidth=0.5, edgecolor='white', legend=False,
    missing_kwds={'color': '#eeeeee'}, ax=axes[0],
    vmin=0, vmax=gdf_latest['bikePed_share'].quantile(0.95)
)
add_colorbar(fig, axes[0], 'YlGn',
             0, gdf_latest['bikePed_share'].quantile(0.95),
             'Bike + Ped Mode Share (%)')
add_tract_labels(axes[0], gdf_latest)
base_map(axes[0], f'Bike + Pedestrian Mode Share — {latest_year}')

In [ ]:
# Right — vehicle
gdf_latest.plot(
    column='vehicle_share', cmap='RdYlGn_r',
    linewidth=0.5, edgecolor='white', legend=False,
    missing_kwds={'color': '#eeeeee'}, ax=axes[1],
    vmin=gdf_latest['vehicle_share'].quantile(0.05), vmax=100
)
add_colorbar(fig, axes[1], 'RdYlGn_r',
             gdf_latest['vehicle_share'].quantile(0.05), 100,
             'Vehicle Mode Share (%)')
add_tract_labels(axes[1], gdf_latest)
base_map(axes[1], f'Vehicle Dominance — {latest_year}')

In [ ]:
fig.suptitle(
    f'Mode Share Comparison by Census Tract — {latest_year}\n'
    f'Bainbridge Island & Kitsap County  |  Source: StreetLight Active Transportation',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'ModeShare_Comparison_{latest_year}.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved ModeShare_Comparison_{latest_year}.pdf")

=============================================================================
ANALYSIS 4: YEAR-OVER-YEAR CHANGE IN BIKE+PED SHARE
=============================================================================

In [ ]:
print("Analysis 4: Year-over-year change maps...")

In [ ]:
years = sorted(wide['Year'].unique())

In [ ]:
if len(years) >= 2:
    first_year = years[0]
    last_year  = years[-1]

    gdf_first = merge_year(first_year)[['tract_id', 'tract_label', 'geometry',
                                         'bikePed_share', 'vehicle_share']].copy()
    gdf_last  = merge_year(last_year) [['tract_id', 'tract_label', 'geometry',
                                         'bikePed_share', 'vehicle_share']].copy()

    gdf_change = gdf_first[['tract_id', 'tract_label', 'geometry']].copy()
    gdf_change = gdf_change.merge(
        gdf_first[['tract_id', 'bikePed_share']].rename(columns={'bikePed_share': 'bp_first'}),
        on='tract_id'
    )
    gdf_change = gdf_change.merge(
        gdf_last[['tract_id', 'bikePed_share']].rename(columns={'bikePed_share': 'bp_last'}),
        on='tract_id'
    )
    gdf_change['bp_change'] = gdf_change['bp_last'] - gdf_change['bp_first']

    # Diverging colormap centered at 0
    max_abs = gdf_change['bp_change'].abs().quantile(0.95)

    fig, ax = plt.subplots(figsize=(14, 12))
    gdf_change.plot(
        column='bp_change',
        cmap='RdYlGn',
        linewidth=0.5,
        edgecolor='white',
        legend=False,
        missing_kwds={'color': '#eeeeee'},
        ax=ax,
        vmin=-max_abs,
        vmax=max_abs
    )
    add_colorbar(fig, ax, 'RdYlGn', -max_abs, max_abs,
                 f'Change in Bike+Ped Mode Share (pp)\n{first_year} → {last_year}')
    add_tract_labels(ax, gdf_change)
    base_map(ax,
             f'Change in Bike + Pedestrian Mode Share\n{first_year} → {last_year}',
             note='Green = mode share increased   Red = mode share declined   White = little change')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'BikePed_Change_{first_year}_{last_year}.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved BikePed_Change_{first_year}_{last_year}.pdf")

=============================================================================
ANALYSIS 5: SMALL MULTIPLES — ONE MAP PER YEAR
=============================================================================

In [ ]:
print("Analysis 5: Small multiples by year...")

In [ ]:
ncols    = 3
nrows    = int(np.ceil(len(years) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                          figsize=(8 * ncols, 8 * nrows))
axes = axes.flatten()

In [ ]:
# Shared scale across all years
global_max = wide['bikePed_share'].quantile(0.95)
global_min = 0

In [ ]:
for idx, year in enumerate(years):
    ax      = axes[idx]
    gdf_yr  = merge_year(year)

    gdf_yr.plot(
        column='bikePed_share',
        cmap='YlGn',
        linewidth=0.3,
        edgecolor='white',
        legend=False,
        missing_kwds={'color': '#eeeeee'},
        ax=ax,
        vmin=global_min,
        vmax=global_max
    )
    add_tract_labels(ax, gdf_yr, fontsize=4.5)
    ax.set_axis_off()
    ax.set_title(str(year), fontsize=12, fontweight='bold')

    # Annotate top 3 tracts
    top3 = gdf_yr.nlargest(3, 'bikePed_share')
    for _, row in top3.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue
        pt = row.geometry.centroid
        ax.annotate(f"★ {row['bikePed_share']:.1f}%",
                    xy=(pt.x, pt.y), xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=5, color='#1a5c1a', fontweight='bold')

In [ ]:
for idx in range(len(years), len(axes)):
    axes[idx].set_visible(False)

In [ ]:
# Shared colorbar
sm = ScalarMappable(cmap='YlGn', norm=Normalize(vmin=global_min, vmax=global_max))
sm.set_array([])
fig.colorbar(sm, ax=axes[:len(years)], fraction=0.01, pad=0.02,
             label='Bike + Pedestrian Mode Share (%)')

In [ ]:
fig.suptitle(
    'Bike + Pedestrian Mode Share by Year — Bainbridge Island & Kitsap County\n'
    '★ = Top 3 tracts per year   Shared color scale across all years',
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'BikePed_SmallMultiples_ByYear.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  Saved BikePed_SmallMultiples_ByYear.pdf")

=============================================================================
ANALYSIS 6: TOP / BOTTOM TRACTS SUMMARY TABLE
=============================================================================

In [ ]:
print("Analysis 6: Top/bottom tract summary...")

In [ ]:
summary = wide[wide['Year'] == latest_year][
    ['Census_Tract_ID', 'bikePed_share', 'vehicle_share',
     'Mode_Share_bicycle', 'Mode_Share_pedestrian']
].copy()
summary['tract_label'] = summary['Census_Tract_ID'].apply(fips_to_label)
summary = summary.sort_values('bikePed_share', ascending=False).reset_index(drop=True)

In [ ]:
print(f"\n  Top 10 tracts by Bike+Ped mode share ({latest_year}):")
print(summary[['tract_label', 'bikePed_share', 'vehicle_share',
               'Mode_Share_bicycle', 'Mode_Share_pedestrian']].head(10).to_string(index=False))

In [ ]:
print(f"\n  Bottom 10 tracts (highest vehicle dominance):")
print(summary[['tract_label', 'bikePed_share', 'vehicle_share']].tail(10).to_string(index=False))

In [ ]:
summary.to_csv(os.path.join(OUTPUT_DIR, f'ModeShare_TractSummary_{latest_year}.csv'), index=False)

In [ ]:
# Bar chart — top 15 and bottom 15
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

In [ ]:
top15 = summary.head(15)
bot15 = summary.tail(15)

In [ ]:
axes[0].barh(top15['tract_label'], top15['bikePed_share'], color='#27ae60')
axes[0].set_xlabel('Bike + Ped Mode Share (%)')
axes[0].set_title(f'Top 15 Tracts — Highest Bike+Ped Share\n{latest_year}', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

In [ ]:
axes[1].barh(bot15['tract_label'], bot15['vehicle_share'], color='#e74c3c')
axes[1].set_xlabel('Vehicle Mode Share (%)')
axes[1].set_title(f'Top 15 Tracts — Highest Vehicle Dominance\n{latest_year}', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

In [ ]:
plt.suptitle('Mode Share Extremes by Census Tract', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'ModeShare_TopBottom_{latest_year}.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  Saved ModeShare_TopBottom bar chart.")

In [ ]:
print(f"\nAll outputs saved to: {OUTPUT_DIR}")